# PolicyEngine numbers for Society in Silico

This notebook reproduces the chapter 6 household marginal rates and chapters 6/13 CTC full-refundability score. It uses actual managed PolicyEngine 5.3.0, its pinned US 1.764.6 model, and certified Microcosm Build P data. See [environment instructions](README.md) and [the computation module](../pe_computed_numbers.py).

Run all cells from the pinned environment. Household calculations require no microdata. The national cell requires the certified H5 **already available locally**. It does not download data or invoke remote compute. Estimates are static; this notebook does not estimate behavioral responses or confidence intervals.

In [1]:
from pathlib import Path
import json
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/pe_computed_numbers.py").is_file())
sys.path.insert(0, str(root / "scripts"))
from pe_computed_numbers import run_households, run_national, OUTPUT
from pe_numbers_report import figures, household_summary, write_summary


## Household assumptions

Both single parents are 35, have only wage income, pay $18,000 annual rent, and have no housing voucher. North Carolina: children aged 8 (SSI disability) and 5. New York: one child aged 8 (SSI disability), outside NYC. Parent/dependent roles and the number of own children are explicit. All people have `is_ssi_ineligible_spouse=False`: none is married to an SSI claimant. This matters because the convenience API puts the family in one marital unit. Parental SSI deeming still operates.

Unspecified inputs use pinned-model defaults, including other program take-up; health-benefit valuations are excluded from net income. Net income includes modeled non-cash benefits and is not a cash bank balance. The full curves retain benefit cliffs. Marginal rates equal `1 - change in net income / change in earnings`, using $100 forward differences. The chapter's $38,000–$39,000 example uses the full $1,000 interval to reduce small dollar-rounding artifacts.

The comparator is a single 45-year-old NYC resident receiving $30 million in long-term capital gains and an additional $10,000. That assumption models the treatment of qualifying long-term gains, not every kind of carried interest.

In [2]:
households = run_households()
print(json.dumps(household_summary(households), indent=2))


Computing nc_two_children earnings curve
Computing ny_one_child earnings curve
Computing NYC long-term capital-gains increment
Computing household CTC full refundability
Computing the 2020–2022 household CTC comparison
{
  "nc_38000_to_39000": {
    "earnings_from": 38000,
    "earnings_to": 39000,
    "mtr": 0.927,
    "changes": {
      "household_net_income": 73.0,
      "household_tax": 426.99951171875,
      "household_benefits": -500.0,
      "household_health_costs": 0.0,
      "household_health_benefits": 0.0,
      "income_tax": 310.599609375,
      "employee_payroll_tax": 76.5,
      "state_income_tax": 39.9000244140625,
      "eitc": -210.599609375,
      "refundable_ctc": -100.0,
      "income_tax_capped_non_refundable_credits": 100.0,
      "snap": 0.0,
      "ssi": -499.9990234375,
      "spousal_deemed_income": 0.0
    }
  },
  "ny_30000_to_30100": {
    "earnings_from": 30000,
    "earnings_to": 30100,
    "mtr": 1.048046875,
    "changes": {
      "household_net_income

## CTC receipt and historical comparison

The CTC case has a North Carolina parent aged 35 and children aged 2 and 4. The 2026 reform changes only `gov.irs.credits.ctc.refundable.fully_refundable` to `True`. It preserves the $2,200 maximum and high-income phase-out. The reform credit is $4,400 for this family throughout the displayed range.

The pinned model's `ctc` and `non_refundable_ctc` are pre-limitation entitlements. Receipt here is the refundable portion plus *applied* nonrefundable credits, after checking that CTC is this household's only nonrefundable credit. At $15,000, the baseline refundable credit is 15% × ($15,000 − $2,500) = $1,875. The historical comparison holds ages and earnings fixed across 2020, 2021, and 2022; it isolates the credit, not the total change in household income between those years.

In [3]:
for path in figures(households):
    print(path.relative_to(root))


scripts/results/pe-computed-numbers/household-marginal-rates.png
scripts/results/pe-computed-numbers/ctc-full-refundability.png


![Household marginal rates](../results/pe-computed-numbers/household-marginal-rates.png)

The left panel shows all computed earnings points; the right focuses on $35,000–$40,000. The dashed line is the separately computed NYC capital-gains rate.

![CTC full refundability](../results/pe-computed-numbers/ctc-full-refundability.png)

## National rescore

This cell performs the full baseline/reform run and can take several minutes and substantial memory. `run_national` resolves the certified dataset by name through `pe.us.managed_microsimulation()`, verifies its local hash, and records provenance. It computes one simulation at a time.

Every population output remains a MicroSeries. Dollar aggregates map to households; poverty flags map to people and are filtered to children before weighted `.mean()` and `.sum()`. No manual person/tax-unit/SPM weights are fetched. Fiscal cost follows PolicyEngine 5.3.0's budget-accounting components: changes in taxes and non-health benefits, plus federal/state health-program costs once. The household resource identity is checked separately and earnings are held fixed. The government health-cost interaction is reported, not assumed to be zero. This is a total static fiscal change, not automatically a federal-only score. Federal income-tax revenue loss is reported separately.

In [4]:
national = run_national()
summary = write_summary(households, national)
print(json.dumps(summary["national"], indent=2))
print("Certified dataset:", national["provenance"]["us_release"]["build_id"])
print("Dataset SHA256:", national["provenance"]["us_release"]["certified_data_artifact"]["sha256"])


Computing national baseline using the certified local bundle
Computing national reform using the certified local bundle
{
  "net_fiscal_cost": 20868288652.260174,
  "federal_income_tax_revenue_loss": 20885352314.074844,
  "poverty": {
    "child_population": 77947677.24210365,
    "baseline": {
      "child_poverty_rate": 0.17017327290983747,
      "children_in_poverty": 13264611.352008432
    },
    "reform": {
      "child_poverty_rate": 0.15122069671750843,
      "children_in_poverty": 11787302.06006239
    },
    "percentage_point_reduction": 1.895257619232904,
    "children_lifted_out": 1477309.2919460423
  }
}
Certified dataset: populace-us-2024-buildp-sparse-rmloss100-cae8640-20260728T011454Z
Dataset SHA256: 48b9d479fb4fd1c3537f9383ce4697d130b6f618658409d74f6233c43b994c7e


## Validation and limits

`tests/test_pe_computed_numbers.py` checks unequal-weight child poverty arithmetic, adult exclusion, index alignment, preservation of mismatched local data, CTC receipt, the SSA 2026 SSI parental-deeming boundary, and reconciliation of every marginal-rate point to taxes and benefits. The [SSA 2026 child SSI table](https://www.ssa.gov/ssi/text-child-ussi.htm) puts a one-parent family with one ineligible child below $4,598 monthly earned income for possible eligibility; the North Carolina curve crosses that boundary.

A static estimate is conditional on the pinned model, population weights, input data and take-up conventions. This run neither replicates the old dense Enhanced CPS score nor supplies a confidence interval. No claim about a pessimistic scenario is supported without actually calculating it. See the JSON receipts for exact versions, data identity, parameters, outputs and execution times.